# Building the balanced 400K incident corpus

This notebook is the executable audit companion for the corpus build. The
source adapters and release checks live in `src/incident_pipeline`; this
notebook reconciles their configuration, raw-file manifest, build reports,
and final SQLite database without silently rebuilding or downloading data.

The target contains exactly **400,000 English reports**: 50,000 records in
each of eight mutually exclusive case types. Descriptions are source-authored
narratives or deterministic renderings of authentic source fields. They are
not generated by an AI model.

## 1. Reproducible setup

Run with the repository `.venv` kernel. The root locator keeps paths stable
whether the kernel starts in the repository or in `notebooks/`.

In [ ]:
from __future__ import annotations

import json
import os
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "incident_pipeline").is_dir() and (
            candidate / "config" / "pipeline.json"
        ).is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root")


PROJECT_ROOT = locate_project_root(Path.cwd().resolve())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
CONFIG_PATH = PROJECT_ROOT / "config" / "pipeline.json"
DB_PATH = Path(
    os.environ.get(
        "HSSE_DATABASE_PATH",
        str(PROCESSED / "hsse_incidents.sqlite"),
    )
).resolve()

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

## 2. Confirm the exact taxonomy and quotas

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
quota_table = pd.DataFrame(
    [
        {
            "Case Type": case_type,
            "Quota": quota,
            "Source adapter": config["case_type_sources"][case_type],
        }
        for case_type, quota in config["case_type_quotas"].items()
    ]
)
display(quota_table)
assert quota_table["Quota"].between(50_000, 100_000).all()
assert quota_table["Quota"].sum() == config["dataset_size"] == 400_000

## 3. Audit the downloaded source snapshot

`source_download_manifest.json` records the exact URL, local path, size, and
SHA-256 digest of every raw input. The build is reproducible against this
snapshot even if an agency later updates a mutable download URL.

In [ ]:
manifest_path = RAW / "source_download_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_table = pd.DataFrame(manifest["files"])
manifest_table["MiB"] = (manifest_table["bytes"] / 2**20).round(1)
manifest_table["SHA-256"] = manifest_table["sha256"].str[:16] + "…"
display(
    manifest_table[
        ["source", "relative_path", "MiB", "SHA-256", "landing_page"]
    ]
)
assert not manifest.get("missing_files")
assert len(manifest_table) == 24

## 4. Rebuild commands

Run these commands from the VS Code terminal when raw files or pipeline rules
change. `rebuild` creates the JSON and provenance sidecar, runs release-blocking
validation, and atomically replaces SQLite only after the checks pass.

```bash
.venv/bin/python -m incident_pipeline.cli download
.venv/bin/python -m incident_pipeline.cli build --config config/pipeline_pilot.json
.venv/bin/python -m incident_pipeline.cli validate --config config/pipeline_pilot.json \
  --report reports/pilot_data_quality_report.json
.venv/bin/python -m incident_pipeline.cli rebuild --config config/pipeline.json
```

The first command reuses existing downloads unless `--force` is supplied.

## 5. Reconcile generated artifacts and reports

In [ ]:
expected_files = [
    PROCESSED / "master_400K.json",
    PROCESSED / "master_400K_with_provenance.jsonl",
    DB_PATH,
    REPORTS / "build_report.json",
    REPORTS / "data_quality_report.json",
]
artifact_table = pd.DataFrame(
    [
        {
            "Artifact": str(path.relative_to(PROJECT_ROOT)),
            "Exists": path.is_file(),
            "MiB": round(path.stat().st_size / 2**20, 1)
            if path.is_file()
            else None,
        }
        for path in expected_files
    ]
)
display(artifact_table)
assert artifact_table["Exists"].all()

build_report = json.loads(
    (REPORTS / "build_report.json").read_text(encoding="utf-8")
)
quality_report = json.loads(
    (REPORTS / "data_quality_report.json").read_text(encoding="utf-8")
)
print("Build rows:", f"{build_report['dataset_rows']:,}")
print("AI-generated reports:", build_report["ai_generated_reports"])
print("Release validation:", quality_report["status"])
assert build_report["dataset_rows"] == 400_000
assert build_report["ai_generated_reports"] == 0
assert quality_report["status"] == "PASS"

## 6. Verify SQLite grain, schema, and quotas

In [ ]:
uri = f"file:{DB_PATH.as_posix()}?mode=ro"
connection = sqlite3.connect(uri, uri=True)

schema = pd.read_sql_query("PRAGMA table_info(incidents)", connection)
display(schema[["cid", "name", "type", "notnull", "pk"]])

checks = {
    "quick_check": connection.execute("PRAGMA quick_check").fetchone()[0],
    "incidents": connection.execute(
        "SELECT COUNT(*) FROM incidents"
    ).fetchone()[0],
    "distinct_case_no": connection.execute(
        "SELECT COUNT(DISTINCT case_no) FROM incidents"
    ).fetchone()[0],
    "distinct_text_hash": connection.execute(
        "SELECT COUNT(DISTINCT text_hash) FROM incidents"
    ).fetchone()[0],
    "source_record_keys": connection.execute(
        "SELECT COUNT(*) FROM (SELECT source, source_record_id "
        "FROM incidents GROUP BY source, source_record_id)"
    ).fetchone()[0],
}
display(pd.Series(checks, name="value").to_frame())
assert checks == {
    "quick_check": "ok",
    "incidents": 400_000,
    "distinct_case_no": 400_000,
    "distinct_text_hash": 400_000,
    "source_record_keys": 400_000,
}

In [ ]:
case_counts = pd.read_sql_query(
    '''
    SELECT case_type AS "Case Type", COUNT(*) AS Records
    FROM incidents
    GROUP BY case_type
    ORDER BY case_type
    ''',
    connection,
)
source_case_counts = pd.read_sql_query(
    '''
    SELECT source AS Source, case_type AS "Case Type", COUNT(*) AS Records
    FROM incidents
    GROUP BY source, case_type
    ORDER BY source, case_type
    ''',
    connection,
)
display(case_counts)
display(source_case_counts)
assert set(case_counts["Records"]) == {50_000}
assert case_counts["Records"].sum() == 400_000

## 7. Inspect provenance and model splits

In [ ]:
split_counts = pd.read_sql_query(
    '''
    SELECT dataset_split AS Split, case_type AS "Case Type", COUNT(*) AS Records
    FROM incidents
    GROUP BY dataset_split, case_type
    ORDER BY dataset_split, case_type
    ''',
    connection,
)
display(split_counts.pivot(index="Case Type", columns="Split", values="Records"))

sample = pd.read_sql_query(
    '''
    SELECT case_no, case_type, source, source_record_id, event_date,
           raw_case_type, raw_hazard, raw_severity,
           substr(description, 1, 180) || '…' AS description_preview
    FROM incidents
    ORDER BY sampling_rank
    LIMIT 8
    ''',
    connection,
)
display(sample)

## 8. Interpretation boundaries

- Rows are authentic reports or deterministic source-field renderings, but
  several sources contain allegations or initial notifications that their
  publishers have not independently verified.
- The eight labels are auditable source/rule assignments, not a
  human-adjudicated gold ontology.
- FAA Service Difficulty Reports cover the asset-loss side of the combined
  Asset and Reputation category; they are not evidence of measured reputation
  damage.
- Source quotas are design choices and cannot be interpreted as real-world
  prevalence, country comparisons, or incident rates.
- Review source-specific redistribution terms in `ATTRIBUTION.md` before
  publishing narrative text.

In [ ]:
connection.close()
print("Closed the read-only SQLite connection.")